# 基于MindSpore NLP的paddleocr-vl图像文本读取识别开发
## 基本环境信息：

Python = 3.10

CANN = 8.2.RC1

MindSpore = 2.7.0

MindSpore NLP == 0.5.1

## 其他主要依赖库与版本：

transformers==4.57.1

diffusers==0.35.2

einops

torchvision

PaddleOCR-VL 是一款面向文档解析的 SOTA 且资源高效的模型。其核心组件为 PaddleOCR-VL-0.9B，这是一种紧凑而强大的视觉语言模型（VLM），由 NaViT 风格的动态分辨率视觉编码器与 ERNIE-4.5-0.3B 语言模型组成，以实现精准的元素识别。该创新模型高效支持 109 种语言，并在识别复杂元素（如文本、表格、公式和图表）方面表现出色，同时保持极低的资源消耗。通过在广泛使用的公开基准与内部基准上的全面评测，PaddleOCR-VL 在页级文档解析与元素级识别两方面均达到 SOTA 表现。它显著优于现有方案，对比顶级 VLM 亦具强竞争力，并具备快速的推理速度。这些优势使其非常适合在真实场景中落地部署。
在本案例中，基于Mindspore框架和mindnlp库利用PaddleOCR-VL 0.9B模型实现常用场景下的OCR识别与基础案例开发。

## 表格识别
首先从表格识别（Table Recognition）开始，这里使用 Transformers 接口加载 Hugging Face 上的模型，先加载 PaddleOCR‑VL‑0.9B 多模态模型及其配套的 tokenizer 和 processor，然后从URL读取一张包含文字信息的图片，构造Table Recognition指令并通过对话模板转换为模型可识别的输入格式；接着由 processor 将图像与文本进行联合编码并送入模型，在 Ascend NPU 上执行生成式推理，最后将模型生成的 token 解码为可读文本，得到图片中表格的结构化识别结果。

In [2]:
import mindspore
import mindnlp
from transformers import AutoModel, AutoProcessor, AutoTokenizer
from transformers.image_utils import load_image


model = AutoModel.from_pretrained("lvyufeng/PaddleOCR-VL-0.9B", trust_remote_code=True, dtype=mindspore.float16, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("lvyufeng/PaddleOCR-VL-0.9B")
processor = AutoProcessor.from_pretrained("lvyufeng/PaddleOCR-VL-0.9B", trust_remote_code=True)

image = load_image(
    "https://hf-mirror.com/datasets/hf-internal-testing/fixtures_got_ocr/resolve/main/image_ocr.jpg"
)

query = 'Table Recognition:'
messages = [
    {
        "role": "user",
        "content": query,
    }
]

text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = processor(image, text=text, return_tensors="pt", format=True).to('cuda')
generate_ids = model.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=1024)
print(generate_ids.shape)
decoded_output = processor.decode(
    generate_ids[0], skip_special_tokens=True
)
print(decoded_output)

/home/mindspore/miniconda/envs/jupyter/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/mindspore/miniconda/envs/jupyter/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/home/mindspore/miniconda/envs/jupyter/lib/python3.10/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/mindspore/miniconda/envs/jupyter/lib/python3.10/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/home/mindspore/miniconda/envs/jupyter

config.json: 0.00B [00:00, ?B/s]

configuration_paddleocr_vl.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lvyufeng/PaddleOCR-VL-0.9B:
- configuration_paddleocr_vl.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_paddleocr_vl.py: 0.00B [00:00, ?B/s]

...

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


image_processing.py: 0.00B [00:00, ?B/s]

Keyword argument `format` is not a valid argument for this processor and will be ignored.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


mindtorch.Size([1, 1204])
User: Table Recognition:
R&D QUALITY IMPROVEMENT SUGGESTION/SOLUTION FORM
Name/Phone Ext.: M. Hamann. P. Harper. P. Martinez Date: 9/3/92
Supervisor/Manager: J. S. Wigand
R&D Group: Licensee
Suggestion: Discontinue coal retention analyses on licensee submitted product samples. (Note: Coal Retention testing is not performed by most licensees. Other B&W physical measurements as ends stability and inspection for soft spots in cigarettes are thought to be sufficient measures to assure cigarette physical integrity. The proposed action will increase laboratory productivity.)
Suggested Solution(s): Delete coal retention from the list of standard analyses performed on licensee submitted product samples. Special requests for coal retention testing could still be submitted on an exception basis.
Have you contacted your Manager/Supervisor?
___ Yes
___ No
Manager Comments: Manager, please contact suggester and forward comments to the Quality Council.
qip.wp
597005708


### 输出后处理
可以看到输出了表格里对应的文本段，这样看起来比较麻烦，实际开发中这样读取出来也需要后处理。可以先以模型生成的 decoded_output 作为原始文本输入，利用正则表达式封装的 extract 方法，从文本中按字段规则依次抽取姓名、日期、主管、研发组、建议内容、解决方案、经理评语和文档编号等关键信息，组织成一个字典结构；随后将该结构化结果打印出来，并通过 json.dumps 转换为格式化的 JSON 输出，在保留中文字符的同时进行缩进美化，最终得到可直接用于存储或接口传输的结构化数据。
将原来的长文本转化成可读性更强、可操作性更强的数据格式。

In [3]:
import re
import json

text = decoded_output

def extract(pattern, text):
    m = re.search(pattern, text, re.S)
    return m.group(1).strip() if m else None

data = {
    "name": extract(r"Name/Phone Ext\.\:\s*(.*?)\s*Date\:", text),
    "date": extract(r"Date\:\s*([0-9/]+)", text),
    "supervisor": extract(r"Supervisor/Manager\:\s*(.*?)\s*R&D Group\:", text),
    "group": extract(r"R&D Group\:\s*(.*?)\s*Suggestion\:", text),
    "suggestion": extract(r"Suggestion\:\s*(.*?)\s*Suggested Solution", text),
    "solution": extract(r"Suggested Solution\(s\)\:\s*(.*?)\s*Have you contacted", text),
    "manager_comments": extract(r"Manager Comments\:\s*(.*?)\s*(qip\.wp|\n\d{6,}$)", text),
    "document_id": extract(r"\n(\d{6,})$", text)
}


print(data)
print('#'*100)
# JSON 输出
json_output = json.dumps(
    data,
    ensure_ascii=False,  #  保留中文
    indent=4,             # 缩进美化
)

print(json_output)

{'name': 'M. Hamann. P. Harper. P. Martinez', 'date': '9/3/92', 'supervisor': 'J. S. Wigand', 'group': 'Licensee', 'suggestion': 'Discontinue coal retention analyses on licensee submitted product samples. (Note: Coal Retention testing is not performed by most licensees. Other B&W physical measurements as ends stability and inspection for soft spots in cigarettes are thought to be sufficient measures to assure cigarette physical integrity. The proposed action will increase laboratory productivity.)', 'solution': 'Delete coal retention from the list of standard analyses performed on licensee submitted product samples. Special requests for coal retention testing could still be submitted on an exception basis.', 'manager_comments': 'Manager, please contact suggester and forward comments to the Quality Council.', 'document_id': '597005708'}
####################################################################################################
{
    "name": "M. Hamann. P. Harper. P. Martinez",


### 数据库操作
实际从图片等提取出文本数据后，往往要进行存储，后续可以接RAG等操作，形成一套完整的工作流。这里以sqlite3为例，将读取并整理的文本数据进行入库操作。

In [4]:
import sqlite3
import json

conn = sqlite3.connect("ocr_records.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS raw_json (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    document_id TEXT,
    payload TEXT
)
""")

cursor.execute("""
INSERT INTO raw_json (document_id, payload)
VALUES (?, ?)
""", (
    data["document_id"],
    json.dumps(data, ensure_ascii=False)
))

conn.commit()
conn.close()

存储后也可以以相同的逻辑进行读取。

In [5]:
import sqlite3
import json

conn = sqlite3.connect("ocr_records.db")
cursor = conn.cursor()

cursor.execute(
    "SELECT payload FROM raw_json WHERE document_id = ?",
    ("597005708",)
)

row = cursor.fetchone()
conn.close()

if row:
    data = json.loads(row[0])
    print(data)

{'name': 'M. Hamann. P. Harper. P. Martinez', 'date': '9/3/92', 'supervisor': 'J. S. Wigand', 'group': 'Licensee', 'suggestion': 'Discontinue coal retention analyses on licensee submitted product samples. (Note: Coal Retention testing is not performed by most licensees. Other B&W physical measurements as ends stability and inspection for soft spots in cigarettes are thought to be sufficient measures to assure cigarette physical integrity. The proposed action will increase laboratory productivity.)', 'solution': 'Delete coal retention from the list of standard analyses performed on licensee submitted product samples. Special requests for coal retention testing could still be submitted on an exception basis.', 'manager_comments': 'Manager, please contact suggester and forward comments to the Quality Council.', 'document_id': '597005708'}


# 通用函数构建
上述案例是对输入的图片直接进行处理，实际开发中可能不太方便，这时候就可以封装成一个函数，后续也方便拓展成一个Agent完成具体功能。

In [6]:
def run_vision_task(
    image_url,
    query,
    model,
    processor,
    tokenizer,
    device="cuda",
    max_new_tokens=1024,
    do_sample=False,
    num_beams=1,
):
    """
    通用视觉-语言推理函数（OCR / 表格识别等）
    """
    # 加载图片
    image = load_image(image_url)

    # 构造对话
    messages = [
        {
            "role": "user",
            "content": query,
        }
    ]

    # 构造模型输入
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = processor(
        image,
        text=text,
        return_tensors="pt",
        format=True
    ).to(device)

    # 推理
    generate_ids = model.generate(
        **inputs,
        do_sample=do_sample,
        num_beams=num_beams,
        max_new_tokens=max_new_tokens
    )

    # 解码输出
    decoded_output = processor.decode(
        generate_ids[0],
        skip_special_tokens=True
    )

    return {
        "token_shape": generate_ids.shape,
        "text": decoded_output
    }

这里选取paddle官方示例图片进行演示，提取一张报刊的文本，不过输出有些粗糙。

In [7]:
ocr_result = run_vision_task(
    image_url="https://paddle-model-ecology.bj.bcebos.com/paddlex/imgs/demo_image/paddleocr_vl_demo.png",
    query="OCR:",
    model=model,
    processor=processor,
    tokenizer=tokenizer
)

print(ocr_result["token_shape"])
print(ocr_result["text"])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


mindtorch.Size([1, 3680])
User: OCR:
助力双方交往
助力双方交往
本报记者 沈小晓
任彦 黄培昭
搭建友谊桥梁
身着中国传统民族服装的厄立特里亚青年依次登台表演中国民族舞、现代舞、扇子舞等，曼妙的舞姿赢得现场观众阵阵掌声。这是日前厄立特里亚高等教育与研究院孔子学院(以下简称"厄特孔院")举办"喜迎新年"中国歌舞比赛的场景。
中国和厄立特里亚传统友谊深厚。近年来，在高质量共建"一带一路"框架下，中厄两国人文交流不断深化，互利合作的民意基础日益深厚。
“学好中文，我们的未来不是梦”
“鲜花曾告诉我你怎样走过，大地知道你心中的每一个角落……”厄立特里亚阿斯马拉大学综合楼二层，一阵优美的歌声在走廊里回响。循着熟悉的旋律轻轻推开一间教室的门，学生们正跟着老师学唱中文歌曲《同一首歌》。
这是厄特孔院阿斯马拉大学教学点的一节中文歌曲课。为了让学生们更好地理解歌词大意，老师尤斯拉·穆罕默德萨尔·侯赛因逐字翻译和解释歌词。随着伴奏声响起，学生们边唱边随着节拍摇动身体，现场气氛热烈。
“这是中文歌曲初级班，共有32人。学生大部分来自首都阿斯马拉的中小学，年龄最小的仅有6岁。”尤斯拉告诉记者。
尤斯拉今年23岁，是厄立特里亚一所公立学校的艺术老师。她12岁开始在厄特孔院学习中文，在2017年第十届"汉语桥"世界中学生中文比赛中获得厄立特里亚赛区第一名，并和同伴代表厄立特里亚前往中国参加决赛，获得团体优胜奖。2022年起，尤斯拉开始在厄特孔院兼职教授中文歌曲，每周末两个课时。"中国文化博大精深，我希望我的学生们能够通过中文歌曲更好地理解中国文化。”她说。
“姐姐，你想去中国吗?”“非常想！我想去看故宫、爬长城。”尤斯拉的学生中有一对能歌善舞的姐妹，姐姐露娅今年15岁，妹妹莉娅14岁，两人都已在厄特孔院学习多年，中文说得格外流利。
露娅对记者说：“这些年来，怀着对中文和中国文化的热爱，我们姐妹俩始终相互鼓励，一起学习。我们的中文一天比一天好，还学会了中文歌和中国舞。我们一定要到中国去。学好中文，我们的未来不是梦！”
据厄特孔院中方院长黄鸣飞介绍，这所孔院成立于2013年3月，由贵州财经大学和
厄立特里亚高等教育与研究院合作建立，开设了中国语言课程和中国文化课程，注册学生2万余人次。10余年来，厄特孔院已成为当地民众了解中国的一扇窗口。
黄鸣飞表示，随着来学

同样的，可以对输出的文本进行清洗操作，并且结构化输出，这里以类似md格式输出文章。

In [8]:
import re
from typing import List


# ---------- 基础工具 ----------

def clean_lines(text: str) -> List[str]:
    """
    基础清洗：去空行、去首尾空格
    """
    return [l.strip() for l in text.splitlines() if l.strip()]


def merge_sentences(lines: List[str]) -> List[str]:
    """
    合并被 OCR 错误断行的句子
    规则：上一行没有以句末标点结束，就合并
    """
    merged = []
    buf = ""

    for line in lines:
        if not buf:
            buf = line
            continue

        if re.search(r"[。！？！”\"]$", buf):
            merged.append(buf)
            buf = line
        else:
            buf += line

    if buf:
        merged.append(buf)

    return merged


# ---------- 结构判断 ----------

def is_title(line: str) -> bool:
    """
    通用标题判断（弱规则）
    """
    return (
        len(line) <= 20
        and not re.search(r"[，。！？：；]", line)
    )


def split_quotes(line: str):
    """
    拆分引语与正文
    返回：quotes, rest
    """
    quotes = re.findall(r"“[^”]+”", line)
    rest = re.sub(r"“[^”]+”", "", line).strip()
    return quotes, rest


# ---------- 主流程 ----------

def ocr_to_markdown(text: str) -> str:
    lines = clean_lines(text)
    lines = merge_sentences(lines)

    md = []
    title_used = False

    for line in lines:

        # 主标题（只取第一个）
        if not title_used and is_title(line):
            md.append(f"# {line}")
            title_used = True
            continue

        # 处理引语
        if "“" in line and "”" in line:
            quotes, rest = split_quotes(line)

            for q in quotes:
                md.append(f"> {q}")

            if rest:
                md.append(rest)

            continue

        # 普通正文
        md.append(line)

    return "\n\n".join(md)


# ---------- 使用示例 ----------

if __name__ == "__main__":
    markdown = ocr_to_markdown(ocr_result["text"])
    print(markdown)

User: OCR:助力双方交往助力双方交往本报记者 沈小晓任彦 黄培昭搭建友谊桥梁身着中国传统民族服装的厄立特里亚青年依次登台表演中国民族舞、现代舞、扇子舞等，曼妙的舞姿赢得现场观众阵阵掌声。这是日前厄立特里亚高等教育与研究院孔子学院(以下简称"厄特孔院")举办"喜迎新年"中国歌舞比赛的场景。

中国和厄立特里亚传统友谊深厚。近年来，在高质量共建"一带一路"框架下，中厄两国人文交流不断深化，互利合作的民意基础日益深厚。

> “学好中文，我们的未来不是梦”

> “鲜花曾告诉我你怎样走过，大地知道你心中的每一个角落……”

厄立特里亚阿斯马拉大学综合楼二层，一阵优美的歌声在走廊里回响。循着熟悉的旋律轻轻推开一间教室的门，学生们正跟着老师学唱中文歌曲《同一首歌》。

这是厄特孔院阿斯马拉大学教学点的一节中文歌曲课。为了让学生们更好地理解歌词大意，老师尤斯拉·穆罕默德萨尔·侯赛因逐字翻译和解释歌词。随着伴奏声响起，学生们边唱边随着节拍摇动身体，现场气氛热烈。

> “这是中文歌曲初级班，共有32人。学生大部分来自首都阿斯马拉的中小学，年龄最小的仅有6岁。”

尤斯拉告诉记者。

尤斯拉今年23岁，是厄立特里亚一所公立学校的艺术老师。她12岁开始在厄特孔院学习中文，在2017年第十届"汉语桥"世界中学生中文比赛中获得厄立特里亚赛区第一名，并和同伴代表厄立特里亚前往中国参加决赛，获得团体优胜奖。2022年起，尤斯拉开始在厄特孔院兼职教授中文歌曲，每周末两个课时。"中国文化博大精深，我希望我的学生们能够通过中文歌曲更好地理解中国文化。”她说。

> “姐姐，你想去中国吗?”

> “非常想！我想去看故宫、爬长城。”

尤斯拉的学生中有一对能歌善舞的姐妹，姐姐露娅今年15岁，妹妹莉娅14岁，两人都已在厄特孔院学习多年，中文说得格外流利。

> “这些年来，怀着对中文和中国文化的热爱，我们姐妹俩始终相互鼓励，一起学习。我们的中文一天比一天好，还学会了中文歌和中国舞。我们一定要到中国去。学好中文，我们的未来不是梦！”

露娅对记者说：

据厄特孔院中方院长黄鸣飞介绍，这所孔院成立于2013年3月，由贵州财经大学和厄立特里亚高等教育与研究院合作建立，开设了中国语言课程和中国文化课程，注册学生2万余人次。10余年来，厄特孔院已成为当地民众了解中国的一扇窗口。

黄鸣飞表示，随着来学习

## 小结
paddle-ocr 0.9B虽然参数量不大，但效果还是很不错的，也适合个人搭建自己的小工作流进行使用，推荐大家尝试尝试。